In [19]:
import pandas as pd
import re
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer




In [2]:
import sys
!"{sys.executable}" -m pip install nltk

  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)


In [6]:
df = pd.read_csv(r"c:\Users\hp\Downloads\UNI\Spring 2026\Bachelor\Bachelor_Project\data\3DatasetsCombined.csv")

In [7]:
#step 1: lower casing
#ensure text col is a string
df['text'] = df['text'].astype(str)
# Create a new column for preprocessing
df['clean_text'] = df['text'].str.lower()
# Preview
df[['text', 'clean_text']].head()


,text,clean_text
0,So if he hit her in the same way for what she ...,so if he hit her in the same way for what she ...
1,"oh fuck im late, , i missed stupid shit im sur...","oh fuck im late, , i missed stupid shit im sur..."
2,"They get attitudes like bitches too , i think ...","they get attitudes like bitches too , i think ..."
3,Im mad james let me drink that shit... payback...,im mad james let me drink that shit... payback...
4,"Wait.. Who respects Colin..., #mkr","wait.. who respects colin..., #mkr"


In [8]:
#step 2: remove URLs, mentions, and hashtags
def remove_patterns(text):
    text = re.sub(r"http\S+|www\S+", "", text)  # remove URLs
    text = re.sub(r"@\w+", "", text)            # remove mentions
    text = re.sub(r"#(\w+)", r"\1", text)       # remove hashtags but keep the word
    return text

# Apply to clean_text column
df['clean_text'] = df['clean_text'].apply(remove_patterns)

# Preview
df[['clean_text']].head()

,clean_text
0,so if he hit her in the same way for what she ...
1,"oh fuck im late, , i missed stupid shit im sur..."
2,"they get attitudes like bitches too , i think ..."
3,im mad james let me drink that shit... payback...
4,"wait.. who respects colin..., mkr"


In [9]:
#step 3: normalize elongated words (e.g., "soooo" -> "soo")
def normalize_elongation(text):
    # replace repeated characters with 2 occurrences
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

# Apply to clean_text column
df['clean_text'] = df['clean_text'].apply(normalize_elongation)

# Preview
df[['clean_text']].head()

,clean_text
0,so if he hit her in the same way for what she ...
1,"oh fuck im late, , i missed stupid shit im sur..."
2,"they get attitudes like bitches too , i think ..."
3,im mad james let me drink that shit.. payback ...
4,"wait.. who respects colin.., mkr"


In [10]:
#step 4: remove punctuation and extra spaces
def remove_punctuation_and_spaces(text):
    # remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    
    # remove extra spaces
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

# Apply to clean_text column
df['clean_text'] = df['clean_text'].apply(remove_punctuation_and_spaces)

# Preview
df[['clean_text']].head()

,clean_text
0,so if he hit her in the same way for what she ...
1,oh fuck im late i missed stupid shit im sure mkr
2,they get attitudes like bitches too i think so...
3,im mad james let me drink that shit payback is...
4,wait who respects colin mkr


In [12]:
#step5: lemmatization
# download required resources 
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [13]:
#step 5: lemmatization
# initialize lemmatizer
lemmatizer = WordNetLemmatizer()

def apply_lemmatization(text):
    tokens = word_tokenize(text)
    lemmatized = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(lemmatized)

# Apply directly to clean_text
df['clean_text'] = df['clean_text'].apply(apply_lemmatization)

# Preview
df[['clean_text']].head()

,clean_text
0,so if he hit her in the same way for what she ...
1,oh fuck im late i missed stupid shit im sure mkr
2,they get attitude like bitch too i think some ...
3,im mad james let me drink that shit payback is...
4,wait who respect colin mkr


In [14]:
#step6: class imbalance 
df['class'].value_counts()

class
1    27761
0    20288
Name: count, dtype: int64

In [18]:
#step 6: class imbalance: compute class weights
classes = np.unique(df['class'])

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=df['class']
)

# convert to dictionary
class_weight_dict = dict(zip(classes, class_weights))

print(class_weight_dict)

{np.int64(0): np.float64(1.1841729100946372), np.int64(1): np.float64(0.8654047044414827)}


In [20]:
#step 7: TF-IDF vectorization
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2)
)

X = tfidf.fit_transform(df['clean_text'])
y = df['class']

In [23]:
df.to_csv(r"C:\Users\hp\Downloads\UNI\Spring 2026\Bachelor\Bachelor_Project\data/combined_processed_v1.csv", index=False)